<a href="https://colab.research.google.com/github/QaziMahadAhmad/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:**
One row = one `(client_hash_id, content_hash_id, query_hash_id)` triple — search performance
for one specific query, on one specific piece of content, for one client. This is query-level,
not content-level: a single content page can have many rows here (one per query it ranks for).
For the actual review decision in this lane ("which content page should be reviewed first"),
these rows get rolled up to content-level downstream — that rollup happens in the baseline
notebook (ML-07), which works at content grain using the daily-performance table. This notebook
documents the query-level source data that feeds those content-level signals.

**Time Window:**
The data covers a **fixed 90-day window** — `window_start = 2026-04-02` to
`window_end = 2026-06-30` (verified below: identical for every row, not a rolling per-row
window). Within that fixed window, two named sub-windows are tracked: `last30` (the final 30
days, 2026-06-01 to 2026-06-30) and `prev30` (the 30 days before that, 2026-05-02 to
2026-05-31). Note `last30 + prev30` covers 60 of the 90 days — the earliest ~30 days of the
window (`2026-04-02` to roughly `2026-05-01`) are only reflected in the `_90d` aggregates, not
in either named sub-window.


In [6]:
import pandas as pd
import numpy as np

fact_path = "../../fact_content_query_90d (1).parquet"
dim_path = "../../dim_content.parquet"

df_fact = pd.read_parquet(fact_path)
df_dim = pd.read_parquet(dim_path)

print("=== Fact Table Columns and Types ===")
print(df_fact.dtypes)

print(f"\nFact shape: {df_fact.shape}")

print("\n=== Dimension Table Columns and Types ===")
print(df_dim.dtypes)

print(f"\nDimension shape: {df_dim.shape}")

=== Fact Table Columns and Types ===
client_hash_id                       str
content_hash_id                      str
query_hash_id                        str
query_char_count                   int64
query_token_count                  int64
window_start                      object
window_end                        object
impressions_90d                    int64
clicks_90d                         int64
impressions_last30                 int64
clicks_last30                      int64
impressions_prev30                 int64
clicks_prev30                      int64
avg_position_90d                 float64
avg_position_last30              float64
avg_position_prev30              float64
content_total_impressions_90d      int64
content_visible_query_count        int64
rare_query_count                   int64
rare_impressions_share           float64
anonymized_impressions_share     float64
dtype: object

Fact shape: (2414248, 21)

=== Dimension Table Columns and Types ===
client_hash_id    

In [7]:
print("=== Temporal Range Fields ===")
print(df_fact[['window_start', 'window_end']].drop_duplicates())

grain_cols = ['client_hash_id', 'content_hash_id', 'query_hash_id']
dups = df_fact.duplicated(subset=grain_cols).sum()
print(f"\nDuplicates on {grain_cols}: {dups} out of {len(df_fact)} rows")

print(f"Unique Clients: {df_fact['client_hash_id'].nunique()}")
print(f"Unique Contents: {df_fact['content_hash_id'].nunique()}")
print(f"Unique Queries: {df_fact['query_hash_id'].nunique()}")

=== Temporal Range Fields ===
  window_start  window_end
0   2026-04-02  2026-06-30

Duplicates on ['client_hash_id', 'content_hash_id', 'query_hash_id']: 0 out of 2414248 rows
Unique Clients: 52
Unique Contents: 133852
Unique Queries: 1180090


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Target / label (defined once, not "depending")
**`ctr_gap_last30 = expected_ctr(avg_position_last30_bucket) − (clicks_last30 / impressions_last30)`**

A continuous score: how far below (or above) a query-content pair's actual `last30` CTR sits
relative to the typical CTR for its position bucket — the same construct the baseline (ML-07)
already uses, just at query grain instead of content grain. This makes the two notebooks tell
one story instead of two. `last30` is the label period; `prev30` and the static dimension
fields are what would have been known *before* that period, so they're the safe features.

- **Features (known before the `last30` label period):**
  - `query_char_count`, `query_token_count`
  - `impressions_prev30`, `clicks_prev30`, `avg_position_prev30`
  - `content_visible_query_count`, `rare_query_count`, `rare_impressions_share`
  - `search_volume`, `competition`, `cpc`, `backlinks`
  - `char_count`, `word_count`
- **Label (defines `ctr_gap_last30`, not usable as a feature):**
  - `impressions_last30`, `clicks_last30`, `avg_position_last30`
- **Excluded — overlaps the label period, not safe as a feature:**
  - `impressions_90d`, `clicks_90d`, `avg_position_90d`, `content_total_impressions_90d` — each
    of these aggregates spans the full 90 days, which *includes* `last30`. Using them as a
    feature to predict something built from `last30` would leak the label into the input.
  - `provider_used`, `model_used` — backend operational details, not signals about ranking
    quality or the target.
  - `anonymized_impressions_share` — excluded to avoid skew from unresolvable/anonymized
    searches diluting the CTR calculation.
- **Context (identifiers/metadata, not fed to a model):**
  - `client_hash_id`, `content_hash_id`, `query_hash_id`, `keyword_hash_id`, `url_hash_id`
  - `content_type`, `competition_level`, `main_intent`
  - `content_created_date`, `content_updated_date`, `last_optimized_date`

**Why the earlier version of this section was wrong:** it listed `impressions_last30` and
`clicks_last30` as *both* features and label material ("can serve as targets... depending"),
which is a direct contradiction — the same column can't be a safe input and the thing being
predicted at the same time. It also listed the `_90d` aggregates as plain features without
noticing they numerically contain the `last30` label window. Fixed above.


In [8]:
# Validate presence of key classification fields and look for null counts
print("=== Missing values in Fact Table ===")
print(df_fact[['impressions_90d', 'clicks_90d', 'avg_position_90d', 'content_total_impressions_90d']].isnull().sum())

print("\n=== Missing values in Dimension Table ===")
cols_to_check = ['search_volume', 'competition', 'cpc', 'backlinks', 'char_count', 'word_count']
print(df_dim[cols_to_check].isnull().sum())

=== Missing values in Fact Table ===
impressions_90d                  0
clicks_90d                       0
avg_position_90d                 0
content_total_impressions_90d    0
dtype: int64

=== Missing values in Dimension Table ===
search_volume    142622
competition      142622
cpc              142622
backlinks        267474
char_count       177768
word_count       177768
dtype: int64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

We verify the correctness of the distributions, null values, and window alignments in the code cells.

In [9]:
# Validate window consistency
same_window_for_all = df_fact[['window_start', 'window_end']].nunique() == 1
print(f"Are window_start and window_end uniform across the dataset? \n{same_window_for_all}")

# Confirm that impressions_last30 and impressions_prev30 sum-bounds align reasonably
logical_check = (df_fact['impressions_last30'] + df_fact['impressions_prev30'] <= df_fact['impressions_90d']).all()
print(f"Is (last30 + prev30) impressions <= 90d impressions for all rows? {logical_check}")
if not logical_check:
    violations = (df_fact['impressions_last30'] + df_fact['impressions_prev30'] > df_fact['impressions_90d']).sum()
    print(f"Number of violations: {violations}")

Are window_start and window_end uniform across the dataset? 
window_start    True
window_end      True
dtype: bool
Is (last30 + prev30) impressions <= 90d impressions for all rows? True


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data Limits Identified:**
1. **Temporal exclusions:** the fact dataset rolls performance up over 90 days (and named
   30-day sub-windows within it). It cannot capture daily spikes, day-of-week patterns, or
   intra-day fluctuations.
2. **Historical disconnection:** `dim_content` is a static snapshot, not a history — past
   states of content attributes (e.g. backlinks or word count *as of* an earlier date) aren't
   recoverable, only the current snapshot value.
3. **Window overlap (label-relevant):** `last30`, `prev30`, and `_90d` are not independent —
   `_90d` numerically contains `last30`, which is exactly why the `_90d` fields are excluded
   from features rather than just "features" (see Section 2). `last30` and `prev30` themselves
   don't overlap each other, but together they only cover 60 of the 90 days — the first ~30
   days of the window aren't isolated in any named sub-window.
4. **Two source tables, two grains:** this notebook profiles `fact_content_query_90d`
   (query-level). The baseline (ML-07) uses `fact_content_daily_performance`
   (content-level, monthly) for the actual position/CTR ranking signal. They describe
   overlapping but not identical periods and grains — a query-level CTR gap here isn't
   guaranteed to move in lockstep with the content-level monthly CTR gap used in the baseline.
   Treat them as related but separate signals, not interchangeable.
5. **No causal claim:** none of this shows *why* CTR is low for a given query — SERP features,
   branded intent, and seasonal demand shifts aren't in this data and can all produce a low CTR
   that a content rewrite won't fix (see ML-07's top-20 review for concrete examples).


In [10]:
# Check if we have any mismatch between fact and dim content IDs
fact_content_set = set(df_fact['content_hash_id'].unique())
dim_content_set = set(df_dim['content_hash_id'].unique())

missing_in_dim = len(fact_content_set - dim_content_set)
missing_in_fact = len(dim_content_set - fact_content_set)

print(f"Content IDs in Fact but missing in Dim: {missing_in_dim}")
print(f"Content IDs in Dim but missing in Fact: {missing_in_fact}")

Content IDs in Fact but missing in Dim: 0
Content IDs in Dim but missing in Fact: 385754


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.